In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import spikeinterface as si
import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages

from tqdm import tqdm


import sys
import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre


import torch.nn.functional as F
from pathlib import Path


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import time
import pickle
import networkx as nx
from probeinterface import write_probeinterface, read_probeinterface, Probe



In [37]:
def count_array2_in_range_of_array1(array1, array2, threshold=5):

    sorted_array1 = np.sort(array1)
    array2 = np.sort(array2)
    
    lefts = array2 - threshold
    rights = array2 + threshold
    
    left_indices = np.searchsorted(sorted_array1, lefts, side='left')
    
    right_indices = np.searchsorted(sorted_array1, rights, side='right')
    
    has_within_range = right_indices > left_indices
    
    count = np.sum(has_within_range)
    
    return count

def label_array1_based_on_array2(array1, array2, threshold=5):
    array_1 = np.sort(array1)
    sorted_array2 = np.sort(array2)
    
    labels = np.zeros(len(array1), dtype=int)
    
    for i, value in enumerate(array1):
        left = value - threshold
        right = value + threshold
        
        left_index = np.searchsorted(sorted_array2, left, side='left')
        right_index = np.searchsorted(sorted_array2, right, side='right')
        
        if right_index > left_index:
            labels[i] = 1
    
    return labels
def detect_local_maxima_in_window(data, window_size=20, std_multiplier=2):

    """
    在每个滑动窗口范围内检测局部最大值的索引，并确保最大值大于两倍的标准差。

    参数:
    data : numpy.ndarray
        输入数据，形状为 (n_rows, n_columns)。
    window_size : int
        滑动窗口的大小，用于定义局部范围，默认为 20。
    std_multiplier : float
        标准差的倍数，用于筛选局部最大值，默认为 2。

    返回:
    local_maxima_indices : list of numpy.ndarray
        每行局部最大值的索引列表，每个元素是对应行局部最大值的索引数组。
    """
    local_maxima_indices = []

    for row in data:
        maxima_indices = []
        row_std = np.std(row.astype(np.float32))
        threshold = std_multiplier * row_std

        for start in range(0, len(row), window_size):
            end = min(start + window_size, len(row))
            window = np.abs(row[start:end])
            
            if len(window) > 0:
                local_max_index = np.argmax(window)
                local_max_value = window[local_max_index]
                
                if local_max_value > threshold:
                    maxima_indices.append(start + local_max_index)  
        
        local_maxima_indices.extend(maxima_indices)
        local_maxima_indices = list(set(local_maxima_indices))  

    return local_maxima_indices


def cluster_label_array1_based_on_array2(array1, array2, threshold=5):

    """
    根据 array2 的 'time' 和 'cluster' 对 array1 进行标记。
    如果 array1 中的某个值在 threshold 范围内存在于 array2 的 'time' 中，则标记为对应的 'cluster' 值，否则为 0。
    
    参数:
    array1 : numpy.ndarray
        要标记的数组。
    array2 : numpy.ndarray
        包含 'time' 和 'cluster' 的二维数组。
        第一列为 'time'，第二列为 'cluster'。
    threshold : int
        判断范围的阈值。
    
    返回:
    labels : numpy.ndarray
        长度为 len(array1) 的标签数组，值为 array2 中的 'cluster' 或 0。
    """

    array2 = np.array(array2.iloc[:, [5, 1]])
    sorted_indices = np.argsort(array2[:, 0])
    sorted_array2 = array2[sorted_indices]
    
    labels = np.zeros(len(array1), dtype=int)
    
    # 遍历 array1 中的每个元素
    for i, value in enumerate(array1):
        # 计算当前值的范围
        left = value - threshold
        right = value + threshold
        
        left_index = np.searchsorted(sorted_array2[:, 0], left, side='left')
        right_index = np.searchsorted(sorted_array2[:, 0], right, side='right')
        
        # 如果范围内存在值，则标记为对应的 'cluster'
        if right_index > left_index:
            # 获取范围内的第一个匹配值的 'cluster'
            labels[i] = sorted_array2[left_index, 1]
    
    return labels


def label_array1_based_on_array2(array1, array2, threshold=5):

    """
    根据 array2 的值对 array1 进行标记。
    如果 array1 中的某个值在 threshold 范围内存在于 array2 中，则标记为 1，否则为 0。
    
    参数:
    array1 : numpy.ndarray
        要标记的数组。
    array2 : numpy.ndarray
        用于判断的数组。
    threshold : int
        判断范围的阈值。
    
    返回:
    labels : numpy.ndarray
        长度为 len(array1) 的标签数组，值为 0 或 1。
    """
    # 对 array2 进行排序以加速搜索
    sorted_array2 = np.sort(array2)
    
    # 初始化标签数组，默认值为 0
    labels = np.zeros(len(array1), dtype=int)
    
    # 遍历 array1 中的每个元素
    for i, value in enumerate(array1):
        # 计算当前值的范围
        left = value - threshold
        right = value + threshold
        
        # 使用二分搜索判断范围内是否存在值
        left_index = np.searchsorted(sorted_array2, left, side='left')
        right_index = np.searchsorted(sorted_array2, right, side='right')
        
        # 如果范围内存在值，则标记为 1
        if right_index > left_index:
            labels[i] = 1
    
    return labels


def extract_windows(data, indices, window_size=61):
    """
    根据给定的时间点索引提取窗口。
    
    参数:
    data : numpy.ndarray
        输入数据，形状为 (n_channels, time)
    indices : numpy.ndarray
        时间点索引数组，用于指定需要提取窗口的中心点
    window_size : int
        窗口长度，默认为61（对应time-30到time+31）
    
    返回:
    windows : numpy.ndarray
        提取的窗口数据，形状为 (len(indices), n_channels, window_size)
    """
    n_channels, time_length = data.shape
    half_window = window_size // 2

    if np.any(indices < half_window) or np.any(indices >= time_length - half_window):
        raise ValueError("Some indices are out of bounds for the given window size.")

    windows = []
    for idx in indices:
        window = data[:, idx - half_window:idx + half_window + 1]
        windows.append(window)

    windows = np.array(windows)
    return windows

In [4]:
recording_raw = se.read_intan(f"/home/ubuntu/Downloads/grid/M190011_250521_141514_merged_130.rhd", stream_id= '0', ignore_integrity_checks=True)
recording_raw = spre.unsigned_to_signed(recording_raw)
recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

recording_f = recording_f.set_probegroup(probe)

In [5]:
spike_inf = pd.read_csv("/media/ubuntu/sda/duan/script/spike_sorting/spike_inf.tsv", index_col=0, sep='\t')
cluster_inf = pd.read_csv("/media/ubuntu/sda/duan/script/spike_sorting/cluster_inf.csv", index_col=0)

In [ ]:
def create_best_channels_group_to_clusters_dict(cluster_inf_df):
    """
    根据best_channels列创建best_channels组合到clusters的映射字典
    
    Args:
        cluster_inf_df: 包含best_channels列的cluster信息DataFrame
    
    Returns:
        dict: {best_channels_tuple: [cluster_id1, cluster_id2, ...]}
    """
    best_channels_group_to_clusters = {}
    
    for idx, row in cluster_inf_df.iterrows():
        cluster_id = row['cluster_id']
        best_channels_str = row['best_channels']
        
        if pd.notna(best_channels_str) and best_channels_str != 'None':
            try:
                # 解析字符串格式的通道列表，例如 "[1, 2, 3]"
                best_channels = eval(best_channels_str)  # 将字符串转换为列表
                
                # 将通道列表转换为元组作为字典的key（因为列表不能作为字典key）
                best_channels_tuple = tuple(sorted(best_channels))
                
                if best_channels_tuple not in best_channels_group_to_clusters:
                    best_channels_group_to_clusters[best_channels_tuple] = []
                best_channels_group_to_clusters[best_channels_tuple].append(cluster_id)
                    
            except (ValueError, SyntaxError) as e:
                print(f"警告: 无法解析cluster {cluster_id}的best_channels: {best_channels_str}")
                continue
    
    # 对每个通道组合的cluster列表进行排序
    for channels_tuple in best_channels_group_to_clusters:
        best_channels_group_to_clusters[channels_tuple].sort()
    
    return best_channels_group_to_clusters

best_channels_group_dict = create_best_channels_group_to_clusters_dict(cluster_inf)

In [8]:
class SpikeDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]
    

class Spike_Detection_MLP(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size, n_channels, time_window):
        super(Spike_Detection_MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size2, 16)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(16, output_size)
        self.sigmoid = nn.Sigmoid()  

        self.n_channels = n_channels
        self.time_window = time_window
    def forward(self, x):
        x = x.reshape(-1, self.n_channels * self.time_window)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.relu3(x)
        x = self.fc4(x)
        x = self.sigmoid(x)
        return x

In [ ]:
# 测试cell - 只处理第一个best_channels组合
print("=== 测试模式：只处理第一个best_channels组合 ===")

# 获取第一个通道组合
first_channels_tuple, first_cluster_ids = list(best_channels_group_dict.items())[0]
channel_group_id = str(list(first_channels_tuple))
print(f'测试处理通道组合: {channel_group_id}')
print(f'对应的clusters: {first_cluster_ids}')

# 创建保存目录
os.makedirs(f'/media/ubuntu/sda/duan/script/spike_sorting/test_result/{channel_group_id}', exist_ok=True)

# 获取该通道组合对应的通道ID列表（这些是contact_ids，直接作为channel_ids使用）
channel_ids = list(first_channels_tuple)

# 检查这些channel_ids是否在recording_f中存在
valid_channels = []
for ch_id in channel_ids:
    valid_channels.append(recording_f.channel_ids[ch_id])  

print(valid_channels) 

if len(valid_channels) == 0:
    print(f"错误: 通道组合 {channel_ids} 中没有可用通道")
else:
    print(f"使用通道: {valid_channels}")
    
    # 读取所有chunk并计算spike检测比例
    total_frames = 1200 * 30000
    chunk_size = 120000  
    window_size = 91
    half_window = window_size // 2
    
    print(f"开始处理所有chunks，总共 {total_frames} 帧...")
    
    all_valid_indices = []
    all_windows = []
    
    for start_frame in tqdm(range(0, total_frames, chunk_size)):
        end_frame = min(start_frame + chunk_size, total_frames)
        
        data_chunk = recording_f.get_traces(
            start_frame=start_frame,
            end_frame=end_frame,
            channel_ids=valid_channels
        )  # shape: (n_channels, chunk_size)
        
        # 检测spike
        threshold_result = detect_local_maxima_in_window(
            data_chunk.T,  
            std_multiplier=1.5,
            window_size=30
        )
        
        # 调整时间戳到全局坐标系
        threshold_result = np.array(threshold_result) + start_frame
        valid_indices = threshold_result[
            (threshold_result >= start_frame + half_window + 1) & 
            (threshold_result < end_frame - half_window)
        ]
        
        # 提取时间窗
        for idx in valid_indices:
            rel_idx = idx - start_frame
            window = data_chunk.T[:, rel_idx-half_window : rel_idx+half_window+1]
            all_windows.append(window)
        
        all_valid_indices.extend(valid_indices)
    
    all_valid_indices = np.array(all_valid_indices)
    all_windows = np.stack(all_windows) if len(all_windows) > 0 else np.array([])
    
    print(f"总共检测到spike数量: {len(all_valid_indices)}")
    print(f"提取的时间窗数量: {len(all_windows)}")
    
    # 获取对应clusters的spike数据
    spike_inf_temp = spike_inf[spike_inf['cluster_id'].isin(first_cluster_ids)]
    print(f"对应clusters的spike数量: {len(spike_inf_temp)}")
    
    if len(spike_inf_temp) > 0:
        print(f"Spike时间范围: {spike_inf_temp['time'].min()} - {spike_inf_temp['time'].max()}")
        
        # 计算spike_inf中包含在检测到的spike中的比例
        # 使用label_array1_based_on_array2函数进行匹配
        labels = label_array1_based_on_array2(all_valid_indices, spike_inf_temp['time'], threshold=3)
        
        detected_spike_count = np.sum(labels == 1)  # 检测到的spike中被标记为真实spike的数量
        total_detected = len(all_valid_indices)
        total_real_spikes = len(spike_inf_temp)
        
        print(f"\n=== Spike检测统计 ===")
        print(f"检测到的spike总数: {total_detected}")
        print(f"真实spike总数: {total_real_spikes}")
        print(f"检测到的真实spike数量: {detected_spike_count}")
        print(f"检测召回率 (detected_real / total_real): {detected_spike_count / total_real_spikes * 100:.2f}%")
        print(f"检测精确率 (detected_real / total_detected): {detected_spike_count / total_detected * 100:.2f}%")
        
        # 计算时间窗数据的统计信息
        if len(all_windows) > 0:
            print(f"\n=== 时间窗数据统计 ===")
            print(f"时间窗形状: {all_windows.shape}")
            print(f"时间窗数据范围: [{all_windows.min():.3f}, {all_windows.max():.3f}]")
            print(f"时间窗数据均值: {all_windows.mean():.3f}")
            print(f"时间窗数据标准差: {all_windows.std():.3f}")
    else:
        print("警告: 没有找到对应的spike数据")
    
    print("=== 测试完成 ===")


=== 测试模式：只处理第一个best_channels组合 ===
测试处理通道组合: [168, 184, 220, 235]
对应的clusters: [0]
['B-040', 'B-056', 'B-092', 'B-107']
使用通道: ['B-040', 'B-056', 'B-092', 'B-107']
开始处理所有chunks，总共 36000000 帧...


100%|██████████| 300/300 [09:57<00:00,  1.99s/it]


总共检测到spike数量: 2883801
提取的时间窗数量: 2883801
对应clusters的spike数量: 556778
Spike时间范围: 454 - 49595270

=== Spike检测统计 ===
检测到的spike总数: 2883801
真实spike总数: 556778
检测到的真实spike数量: 602342
检测召回率 (detected_real / total_real): 108.18%
检测精确率 (detected_real / total_detected): 20.89%

=== 时间窗数据统计 ===
时间窗形状: (2883801, 4, 91)
时间窗数据范围: [-331.000, 238.000]
时间窗数据均值: -0.055
时间窗数据标准差: 26.143
=== 测试完成 ===


In [ ]:
labels = label_array1_based_on_array2(all_valid_indices, spike_inf_temp['time'], threshold=5)
indices_0 = np.where(labels == 0)[0] 
indices_1 = np.where(labels == 1)[0] 

target_0_count = len(indices_1) 

if len(indices_0) > target_0_count:
    sampled_indices_0 = np.random.choice(indices_0, target_0_count, replace=False)
else:
    sampled_indices_0 = indices_0  

final_indices = np.concatenate([sampled_indices_0, indices_1])
np.random.shuffle(final_indices)

sampled_windows = all_windows[final_indices]
sampled_labels = labels[final_indices]

dataset = SpikeDataset(sampled_windows, sampled_labels)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

batch_size = 1024 
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

hidden_size1 = 256
hidden_size2 = 64
output_size = 1  
device = 'cuda'
input_size = sampled_windows.shape[1] * sampled_windows.shape[2]

criterion = nn.BCELoss()  

In [ ]:
model = Spike_Detection_MLP(input_size, hidden_size1, hidden_size2, 
                                output_size, n_channels=sampled_windows.shape[1], time_window=sampled_windows.shape[2])
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001)

num_epochs = 50
tpr_best = 0
i = 0
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for batch_data, batch_labels in train_loader:
        batch_labels = batch_labels.float().unsqueeze(1)

        batch_data = batch_data.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_data)
        loss = criterion(outputs, batch_labels)

        predicted = (outputs > 0.5).float()  
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    model.eval()
    correct = 0
    total = 0

    true_positive = 0
    true_negative = 0
    false_positive = 0
    false_negative = 0

    with torch.no_grad():
        for batch_data, batch_labels in test_loader:
            batch_labels = batch_labels.float().unsqueeze(1)
            batch_data = batch_data.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model(batch_data)
            predicted = (outputs > 0.5).float()  
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()
            true_positive += ((predicted == 1) & (batch_labels == 1)).sum().item()
            true_negative += ((predicted == 0) & (batch_labels == 0)).sum().item()
            false_positive += ((predicted == 1) & (batch_labels == 0)).sum().item()
            false_negative += ((predicted == 0) & (batch_labels == 1)).sum().item()

    tpr = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
    tnr = true_negative / (true_negative + false_positive) if (true_negative + false_positive) > 0 else 0

    print(f"Epoch {epoch} tpr: {tpr}; tnr: {tnr}")
    print("_" * 60)
    
    # if tpr > tpr_best:
    #     tpr_best = tpr
    #     i = 0
    # else:
    #     i += 1
    #     if i == 3:
    #         break

Epoch 0 tpr: 0.9146825232507102; tnr: 0.852988651113256
____________________________________________________________
Epoch 1 tpr: 0.9056803783054104; tnr: 0.8747545512031151
____________________________________________________________
Epoch 2 tpr: 0.9317344243016505; tnr: 0.8527806436582687
____________________________________________________________
Epoch 3 tpr: 0.9156100671641173; tnr: 0.8783572403234932
____________________________________________________________
Epoch 4 tpr: 0.9207860934666126; tnr: 0.8756032216194628
____________________________________________________________


KeyboardInterrupt: 

In [38]:
# 完整循环处理所有best_channels_group_dict
print("=== 开始处理所有best_channels_group_dict ===")
print(f"总共需要处理 {len(best_channels_group_dict)} 个通道组合")

# 创建主结果保存目录
main_result_dir = '/media/ubuntu/sda/duan/script/spike_sorting/all_results'
os.makedirs(main_result_dir, exist_ok=True)

# 存储所有结果的字典
all_results = {}

# 训练参数
hidden_size1 = 256
hidden_size2 = 64
output_size = 1
device = 'cuda'
num_epochs = 50
batch_size = 1024
window_size = 91
half_window = window_size // 2
chunk_size = 120000
total_frames = 1200 * 30000

# 处理每个通道组合
for idx, (channels_tuple, cluster_ids) in enumerate(best_channels_group_dict.items()):
    channel_group_id = str(list(channels_tuple))
    print(f"\n{'='*80}")
    print(f"处理第 {idx+1}/{len(best_channels_group_dict)} 个通道组合: {channel_group_id}")
    print(f"对应的clusters: {cluster_ids}")
    print(f"{'='*80}")
    
    try:
        # 创建该通道组合的结果保存目录
        result_dir = os.path.join(main_result_dir, f"channels_{channel_group_id.replace(' ', '').replace('[', '').replace(']', '')}")
        os.makedirs(result_dir, exist_ok=True)
        
        # 获取该通道组合对应的通道ID列表
        channel_ids = list(channels_tuple)
        
        # 检查这些channel_ids是否在recording_f中存在
        valid_channels = []
        for ch_id in channel_ids:
            if ch_id < len(recording_f.channel_ids):
                valid_channels.append(recording_f.channel_ids[ch_id])
            else:
                print(f"警告: 通道ID {ch_id} 超出范围，跳过")
                break
        
        if len(valid_channels) == 0:
            print(f"错误: 通道组合 {channel_ids} 中没有可用通道，跳过")
            continue
            
        print(f"使用通道: {valid_channels}")
        
        # 读取所有chunk并计算spike检测
        print(f"开始处理所有chunks，总共 {total_frames} 帧...")
        
        all_valid_indices = []
        all_windows = []
        
        for start_frame in tqdm(range(0, total_frames, chunk_size), desc=f"处理chunks"):
            end_frame = min(start_frame + chunk_size, total_frames)
            
            try:
                data_chunk = recording_f.get_traces(
                    start_frame=start_frame,
                    end_frame=end_frame,
                    channel_ids=valid_channels
                )  # shape: (n_channels, chunk_size)
                
                # 检测spike
                threshold_result = detect_local_maxima_in_window(
                    data_chunk.T,  
                    std_multiplier=1.5,
                    window_size=30
                )
                
                # 调整时间戳到全局坐标系
                threshold_result = np.array(threshold_result) + start_frame
                valid_indices = threshold_result[
                    (threshold_result >= start_frame + half_window + 1) & 
                    (threshold_result < end_frame - half_window)
                ]
                
                # 提取时间窗
                for idx_val in valid_indices:
                    rel_idx = idx_val - start_frame
                    window = data_chunk.T[:, rel_idx-half_window : rel_idx+half_window+1]
                    all_windows.append(window)
                
                all_valid_indices.extend(valid_indices)
                
            except Exception as e:
                print(f"处理chunk时出错: {e}")
                continue
        
        all_valid_indices = np.array(all_valid_indices)
        all_windows = np.stack(all_windows) if len(all_windows) > 0 else np.array([])
        
        print(f"总共检测到spike数量: {len(all_valid_indices)}")
        print(f"提取的时间窗数量: {len(all_windows)}")
        
        # 获取对应clusters的spike数据
        spike_inf_temp = spike_inf[spike_inf['cluster_id'].isin(cluster_ids)]
        print(f"对应clusters的spike数量: {len(spike_inf_temp)}")
        
        if len(spike_inf_temp) == 0:
            print("警告: 没有找到对应的spike数据，跳过此通道组合")
            continue
            
        # 计算spike检测统计
        labels = label_array1_based_on_array2(all_valid_indices, spike_inf_temp['time'], threshold=5)
        
        detected_spike_count = np.sum(labels == 1)
        total_detected = len(all_valid_indices)
        total_real_spikes = len(spike_inf_temp)
        
        detection_recall = detected_spike_count / total_real_spikes * 100 if total_real_spikes > 0 else 0
        detection_precision = detected_spike_count / total_detected * 100 if total_detected > 0 else 0
        
        print(f"\n=== Spike检测统计 ===")
        print(f"检测到的spike总数: {total_detected}")
        print(f"真实spike总数: {total_real_spikes}")
        print(f"检测到的真实spike数量: {detected_spike_count}")
        print(f"检测召回率: {detection_recall:.2f}%")
        print(f"检测精确率: {detection_precision:.2f}%")
        
        # 准备训练数据
        indices_0 = np.where(labels == 0)[0] 
        indices_1 = np.where(labels == 1)[0] 
        
        target_0_count = len(indices_1)
        
        if len(indices_0) > target_0_count:
            sampled_indices_0 = np.random.choice(indices_0, target_0_count, replace=False)
        else:
            sampled_indices_0 = indices_0  
        
        final_indices = np.concatenate([sampled_indices_0, indices_1])
        np.random.shuffle(final_indices)
        
        sampled_windows = all_windows[final_indices]
        sampled_labels = labels[final_indices]
        
        # 创建数据集
        dataset = SpikeDataset(sampled_windows, sampled_labels)
        
        train_size = int(0.8 * len(dataset))
        test_size = len(dataset) - train_size
        train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        
        # 创建模型
        input_size = sampled_windows.shape[1] * sampled_windows.shape[2]
        model = Spike_Detection_MLP(input_size, hidden_size1, hidden_size2, 
                                    output_size, n_channels=sampled_windows.shape[1], time_window=sampled_windows.shape[2])
        model = model.to(device)
        
        optimizer = optim.Adam(model.parameters(), lr=0.0001)
        criterion = nn.BCELoss()
        
        # 训练模型
        print(f"\n开始训练模型...")
        training_history = []
        best_tpr = 0
        best_model_state = None
        patience = 5  # 早停耐心值
        patience_counter = 0  # 早停计数器
        
        for epoch in range(num_epochs):
            # 训练阶段
            model.train()
            train_loss = 0
            train_correct = 0
            train_total = 0
            
            for batch_data, batch_labels in train_loader:
                batch_labels = batch_labels.float().unsqueeze(1)
                batch_data = batch_data.to(device)
                batch_labels = batch_labels.to(device)
                
                outputs = model(batch_data)
                loss = criterion(outputs, batch_labels)
                
                predicted = (outputs > 0.5).float()
                train_total += batch_labels.size(0)
                train_correct += (predicted == batch_labels).sum().item()
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
            
            # 验证阶段
            model.eval()
            test_correct = 0
            test_total = 0
            true_positive = 0
            true_negative = 0
            false_positive = 0
            false_negative = 0
            
            with torch.no_grad():
                for batch_data, batch_labels in test_loader:
                    batch_labels = batch_labels.float().unsqueeze(1)
                    batch_data = batch_data.to(device)
                    batch_labels = batch_labels.to(device)
                    
                    outputs = model(batch_data)
                    predicted = (outputs > 0.5).float()
                    test_total += batch_labels.size(0)
                    test_correct += (predicted == batch_labels).sum().item()
                    
                    true_positive += ((predicted == 1) & (batch_labels == 1)).sum().item()
                    true_negative += ((predicted == 0) & (batch_labels == 0)).sum().item()
                    false_positive += ((predicted == 1) & (batch_labels == 0)).sum().item()
                    false_negative += ((predicted == 0) & (batch_labels == 1)).sum().item()
            
            tpr = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
            tnr = true_negative / (true_negative + false_positive) if (true_negative + false_positive) > 0 else 0
            accuracy = test_correct / test_total if test_total > 0 else 0
            
            # 保存最佳模型和早停机制
            if tpr > best_tpr:
                best_tpr = tpr
                best_model_state = model.state_dict().copy()
                patience_counter = 0  # 重置计数器
                print(f"Epoch {epoch}: 新的最佳TPR={tpr:.4f}, 重置早停计数器")
            else:
                patience_counter += 1
                print(f"Epoch {epoch}: TPR={tpr:.4f} (最佳: {best_tpr:.4f}), 早停计数器: {patience_counter}/{patience}")
            
            training_history.append({
                'epoch': epoch,
                'train_loss': train_loss / len(train_loader),
                'train_accuracy': train_correct / train_total if train_total > 0 else 0,
                'test_accuracy': accuracy,
                'tpr': tpr,
                'tnr': tnr,
                'patience_counter': patience_counter
            })
            
            if epoch % 10 == 0 or epoch == num_epochs - 1:
                print(f"Epoch {epoch}: TPR={tpr:.4f}, TNR={tnr:.4f}, Accuracy={accuracy:.4f}")
            
            # 早停检查
            if patience_counter >= patience:
                print(f"\n早停触发！连续 {patience} 个epoch没有提升，在第 {epoch+1} 个epoch停止训练")
                print(f"最佳TPR: {best_tpr:.4f}")
                break
        
        # 保存最佳模型
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            torch.save(model.state_dict(), os.path.join(result_dir, 'best_model.pth'))
        
        # 计算实际训练的epoch数
        actual_epochs = len(training_history)
        early_stopped = actual_epochs < num_epochs
        
        # 保存结果
        result_summary = {
            'channel_group_id': channel_group_id,
            'channels': valid_channels,
            'cluster_ids': cluster_ids,
            'detection_stats': {
                'total_detected': int(total_detected),
                'total_real_spikes': int(total_real_spikes),
                'detected_real_spikes': int(detected_spike_count),
                'detection_recall': float(detection_recall),
                'detection_precision': float(detection_precision)
            },
            'training_stats': {
                'best_tpr': float(best_tpr),
                'final_tnr': float(tnr),
                'final_accuracy': float(accuracy),
                'total_epochs': num_epochs,
                'actual_epochs': actual_epochs,
                'early_stopped': early_stopped,
                'patience': patience
            },
            'data_stats': {
                'window_shape': sampled_windows.shape,
                'train_samples': len(train_dataset),
                'test_samples': len(test_dataset)
            }
        }
        
        # 保存详细结果
        with open(os.path.join(result_dir, 'result_summary.pkl'), 'wb') as f:
            pickle.dump(result_summary, f)
        
        with open(os.path.join(result_dir, 'training_history.pkl'), 'wb') as f:
            pickle.dump(training_history, f)
        
        # 保存到总结果字典
        all_results[channel_group_id] = result_summary
        
        print(f"通道组合 {channel_group_id} 处理完成，最佳TPR: {best_tpr:.4f}")
        
    except Exception as e:
        print(f"处理通道组合 {channel_group_id} 时出错: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*80}")
print("所有通道组合处理完成！")
print(f"{'='*80}")

# 保存总结果
with open(os.path.join(main_result_dir, 'all_results_summary.pkl'), 'wb') as f:
    pickle.dump(all_results, f)

print(f"结果已保存到: {main_result_dir}")


=== 开始处理所有best_channels_group_dict ===
总共需要处理 52 个通道组合

处理第 1/52 个通道组合: [168, 184, 220, 235]
对应的clusters: [0]
使用通道: ['B-040', 'B-056', 'B-092', 'B-107']
开始处理所有chunks，总共 36000000 帧...


处理chunks: 100%|██████████| 300/300 [10:16<00:00,  2.05s/it]


总共检测到spike数量: 2883801
提取的时间窗数量: 2883801
对应clusters的spike数量: 556778

=== Spike检测统计 ===
检测到的spike总数: 2883801
真实spike总数: 556778
检测到的真实spike数量: 602342
检测召回率: 108.18%
检测精确率: 20.89%

开始训练模型...
Epoch 0: 新的最佳TPR=0.9080, 重置早停计数器
Epoch 0: TPR=0.9080, TNR=0.8637, Accuracy=0.8859
Epoch 1: TPR=0.9010 (最佳: 0.9080), 早停计数器: 1/5
Epoch 2: 新的最佳TPR=0.9132, 重置早停计数器
Epoch 3: TPR=0.9067 (最佳: 0.9132), 早停计数器: 1/5
Epoch 4: 新的最佳TPR=0.9194, 重置早停计数器
Epoch 5: 新的最佳TPR=0.9211, 重置早停计数器
Epoch 6: 新的最佳TPR=0.9241, 重置早停计数器
Epoch 7: TPR=0.9158 (最佳: 0.9241), 早停计数器: 1/5
Epoch 8: TPR=0.9191 (最佳: 0.9241), 早停计数器: 2/5
Epoch 9: TPR=0.9227 (最佳: 0.9241), 早停计数器: 3/5
Epoch 10: TPR=0.9189 (最佳: 0.9241), 早停计数器: 4/5
Epoch 10: TPR=0.9189, TNR=0.8902, Accuracy=0.9046
Epoch 11: TPR=0.9238 (最佳: 0.9241), 早停计数器: 5/5

早停触发！连续 5 个epoch没有提升，在第 12 个epoch停止训练
最佳TPR: 0.9241
通道组合 [168, 184, 220, 235] 处理完成，最佳TPR: 0.9241

处理第 2/52 个通道组合: [136, 191, 200, 207, 236, 252]
对应的clusters: [1]
使用通道: ['B-008', 'B-063', 'B-072', 'B-079', 'B-108', 'B-124']
开始处理所有ch

处理chunks: 100%|██████████| 300/300 [10:18<00:00,  2.06s/it]


总共检测到spike数量: 4148896
提取的时间窗数量: 4148896
对应clusters的spike数量: 387985

=== Spike检测统计 ===
检测到的spike总数: 4148896
真实spike总数: 387985
检测到的真实spike数量: 429992
检测召回率: 110.83%
检测精确率: 10.36%

开始训练模型...
Epoch 0: 新的最佳TPR=0.9271, 重置早停计数器
Epoch 0: TPR=0.9271, TNR=0.8675, Accuracy=0.8972
Epoch 1: 新的最佳TPR=0.9294, 重置早停计数器
Epoch 2: 新的最佳TPR=0.9311, 重置早停计数器
Epoch 3: 新的最佳TPR=0.9382, 重置早停计数器
Epoch 4: TPR=0.9338 (最佳: 0.9382), 早停计数器: 1/5
Epoch 5: 新的最佳TPR=0.9454, 重置早停计数器
Epoch 6: 新的最佳TPR=0.9480, 重置早停计数器
Epoch 7: TPR=0.9444 (最佳: 0.9480), 早停计数器: 1/5
Epoch 8: TPR=0.9384 (最佳: 0.9480), 早停计数器: 2/5
Epoch 9: TPR=0.9333 (最佳: 0.9480), 早停计数器: 3/5
Epoch 10: TPR=0.9433 (最佳: 0.9480), 早停计数器: 4/5
Epoch 10: TPR=0.9433, TNR=0.9045, Accuracy=0.9238
Epoch 11: TPR=0.9458 (最佳: 0.9480), 早停计数器: 5/5

早停触发！连续 5 个epoch没有提升，在第 12 个epoch停止训练
最佳TPR: 0.9480
通道组合 [136, 191, 200, 207, 236, 252] 处理完成，最佳TPR: 0.9480

处理第 3/52 个通道组合: [199, 208, 210, 240, 241, 242]
对应的clusters: [2, 27, 40, 76, 78]
使用通道: ['B-071', 'B-080', 'B-082', 'B-112', 'B-113', 'B-

...

总共检测到spike数量: 4376230
提取的时间窗数量: 4376230
对应clusters的spike数量: 418229

=== Spike检测统计 ===
检测到的spike总数: 4376230
真实spike总数: 418229
检测到的真实spike数量: 485644
检测召回率: 116.12%
检测精确率: 11.10%

开始训练模型...
Epoch 0: 新的最佳TPR=0.9091, 重置早停计数器
Epoch 0: TPR=0.9091, TNR=0.9032, Accuracy=0.9061
Epoch 1: 新的最佳TPR=0.9331, 重置早停计数器
Epoch 2: 新的最佳TPR=0.9412, 重置早停计数器
Epoch 3: TPR=0.9289 (最佳: 0.9412), 早停计数器: 1/5
Epoch 4: TPR=0.9358 (最佳: 0.9412), 早停计数器: 2/5
Epoch 5: 新的最佳TPR=0.9435, 重置早停计数器
Epoch 6: 新的最佳TPR=0.9439, 重置早停计数器
Epoch 7: TPR=0.9381 (最佳: 0.9439), 早停计数器: 1/5
Epoch 8: TPR=0.9426 (最佳: 0.9439), 早停计数器: 2/5
Epoch 9: TPR=0.9317 (最佳: 0.9439), 早停计数器: 3/5
Epoch 10: TPR=0.9352 (最佳: 0.9439), 早停计数器: 4/5
Epoch 10: TPR=0.9352, TNR=0.9204, Accuracy=0.9278
Epoch 11: TPR=0.9335 (最佳: 0.9439), 早停计数器: 5/5

早停触发！连续 5 个epoch没有提升，在第 12 个epoch停止训练
最佳TPR: 0.9439
通道组合 [135, 151, 179, 227, 228, 243] 处理完成，最佳TPR: 0.9439

处理第 51/52 个通道组合: [135, 151, 167, 228, 230, 243]
对应的clusters: [146]
使用通道: ['B-007', 'B-023', 'B-039', 'B-100', 'B-102', 'B-1

处理chunks: 100%|██████████| 300/300 [09:46<00:00,  1.95s/it]


总共检测到spike数量: 4169710
提取的时间窗数量: 4169710
对应clusters的spike数量: 583244

=== Spike检测统计 ===
检测到的spike总数: 4169710
真实spike总数: 583244
检测到的真实spike数量: 697639
检测召回率: 119.61%
检测精确率: 16.73%

开始训练模型...
Epoch 0: 新的最佳TPR=0.8734, 重置早停计数器
Epoch 0: TPR=0.8734, TNR=0.8655, Accuracy=0.8695
Epoch 1: 新的最佳TPR=0.9021, 重置早停计数器
Epoch 2: TPR=0.9015 (最佳: 0.9021), 早停计数器: 1/5
Epoch 3: 新的最佳TPR=0.9074, 重置早停计数器
Epoch 4: TPR=0.9070 (最佳: 0.9074), 早停计数器: 1/5
Epoch 5: TPR=0.9023 (最佳: 0.9074), 早停计数器: 2/5
Epoch 6: 新的最佳TPR=0.9206, 重置早停计数器
Epoch 7: TPR=0.9011 (最佳: 0.9206), 早停计数器: 1/5
Epoch 8: TPR=0.8984 (最佳: 0.9206), 早停计数器: 2/5
Epoch 9: TPR=0.9191 (最佳: 0.9206), 早停计数器: 3/5
Epoch 10: TPR=0.9135 (最佳: 0.9206), 早停计数器: 4/5
Epoch 10: TPR=0.9135, TNR=0.8889, Accuracy=0.9012
Epoch 11: TPR=0.9121 (最佳: 0.9206), 早停计数器: 5/5

早停触发！连续 5 个epoch没有提升，在第 12 个epoch停止训练
最佳TPR: 0.9206
通道组合 [135, 151, 167, 228, 230, 243] 处理完成，最佳TPR: 0.9206

处理第 52/52 个通道组合: [135, 151, 167, 230]
对应的clusters: [147]
使用通道: ['B-007', 'B-023', 'B-039', 'B-102']
开始处理所有chunk

处理chunks: 100%|██████████| 300/300 [09:57<00:00,  1.99s/it]


总共检测到spike数量: 2731644
提取的时间窗数量: 2731644
对应clusters的spike数量: 16067

=== Spike检测统计 ===
检测到的spike总数: 2731644
真实spike总数: 16067
检测到的真实spike数量: 19417
检测召回率: 120.85%
检测精确率: 0.71%

开始训练模型...
Epoch 0: 新的最佳TPR=0.9610, 重置早停计数器
Epoch 0: TPR=0.9610, TNR=0.9173, Accuracy=0.9392
Epoch 1: 新的最佳TPR=0.9851, 重置早停计数器
Epoch 2: 新的最佳TPR=0.9897, 重置早停计数器
Epoch 3: 新的最佳TPR=0.9928, 重置早停计数器
Epoch 4: TPR=0.9923 (最佳: 0.9928), 早停计数器: 1/5
Epoch 5: 新的最佳TPR=0.9941, 重置早停计数器
Epoch 6: TPR=0.9938 (最佳: 0.9941), 早停计数器: 1/5
Epoch 7: TPR=0.9938 (最佳: 0.9941), 早停计数器: 2/5
Epoch 8: TPR=0.9941 (最佳: 0.9941), 早停计数器: 3/5
Epoch 9: TPR=0.9931 (最佳: 0.9941), 早停计数器: 4/5
Epoch 10: TPR=0.9936 (最佳: 0.9941), 早停计数器: 5/5
Epoch 10: TPR=0.9936, TNR=0.9935, Accuracy=0.9936

早停触发！连续 5 个epoch没有提升，在第 11 个epoch停止训练
最佳TPR: 0.9941
通道组合 [135, 151, 167, 230] 处理完成，最佳TPR: 0.9941

所有通道组合处理完成！
结果已保存到: /media/ubuntu/sda/duan/script/spike_sorting/all_results


In [46]:
# Load all_results and extract training metrics
import pickle
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Load all_results
with open('/media/ubuntu/sda/duan/script/spike_sorting/all_results/all_results_summary.pkl', 'rb') as f:
    all_results = pickle.load(f)

print(f"Loaded results for {len(all_results)} channel combinations")

# Extract training metrics data
training_metrics = []

for channel_group_id, result_summary in all_results.items():
    training_data = {
        'channel_group_id': channel_group_id,
        'best_tpr': result_summary['training_stats']['best_tpr'],
        'final_tnr': result_summary['training_stats']['final_tnr'],
        'final_accuracy': result_summary['training_stats']['final_accuracy'],
        'actual_epochs': result_summary['training_stats']['actual_epochs'],
        'early_stopped': result_summary['training_stats']['early_stopped']
    }
    training_metrics.append(training_data)

# Convert to DataFrame
training_df = pd.DataFrame(training_metrics)

print("Training metrics extraction completed!")
print(f"Training metrics shape: {training_df.shape}")


Loaded results for 52 channel combinations
Training metrics extraction completed!
Training metrics shape: (52, 6)


In [64]:
with PdfPages('/media/ubuntu/sda/duan/figure/spike_detection_eval.pdf') as pdf:
    fig, ax = plt.subplots(1, 1, figsize=(5, 3))

    # Prepare data for boxplot
    metrics_data = [
        training_df['best_tpr'] * 100,  # Convert to percentage
        training_df['final_tnr'] * 100,  # Convert to percentage
        training_df['final_accuracy'] * 100  # Convert to percentage
    ]

    labels = ['TPR', 'TNR', 'Accuracy']

    # Create boxplot
    bp = ax.boxplot(metrics_data, labels=labels, patch_artist=True)

    # Set colors for each box
    colors = ['#EA8379', '#7DAEE0', '#B395BD']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(1)

    # Change median line color to black
    for median in bp['medians']:
        median.set_color('black')
        median.set_linewidth(1)

    # Customize the plot
    ax.set_title('Model Performance Metrics', fontsize=14, fontweight='bold')
    ax.set_ylabel('Percentage (%)', fontsize=12)
    ax.set_xlabel('Metrics', fontsize=12)
    ax.set_ylim(0, 105)
    ax.grid(False)


    plt.tight_layout()
    pdf.savefig()
    plt.close()


In [72]:
# 修改模型类以支持特征提取
class Spike_Detection_MLP_with_features(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size, n_channels, time_window):
        super(Spike_Detection_MLP_with_features, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size2, 16)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(16, output_size)
        self.sigmoid = nn.Sigmoid()  

        self.n_channels = n_channels
        self.time_window = time_window
        
    def forward(self, x):
        x = x.reshape(-1, self.n_channels * self.time_window)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.relu3(x)
        # 返回最后一层特征（fc3的输出）和最终预测结果
        features = x
        x = self.fc4(x)
        x = self.sigmoid(x)
        return x, features
    
    def extract_features(self, x):
        """提取最后一层特征"""
        x = x.reshape(-1, self.n_channels * self.time_window)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.relu3(x)
        return x

print("模型类已更新，支持特征提取功能")


模型类已更新，支持特征提取功能


In [73]:
# 随机选择3个group进行可视化验证
import random
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# 设置随机种子以确保结果可重现
random.seed(42)
np.random.seed(42)

# 从所有结果中随机选择3个group
all_channel_groups = list(all_results.keys())
selected_groups = random.sample(all_channel_groups, min(3, len(all_channel_groups)))

print(f"随机选择的3个group:")
for i, group in enumerate(selected_groups):
    print(f"{i+1}. {group}")

print(f"\n开始为这3个group创建可视化验证...")


随机选择的3个group:
1. [22, 38, 103, 114, 115, 119]
2. [190, 201, 205, 237, 239, 255]
3. [136, 191, 200, 207, 236, 252]

开始为这3个group创建可视化验证...


In [74]:
# 为选中的3个group创建可视化验证
def create_visualization_for_group(channel_group_id, result_dir):
    """为单个group创建可视化"""
    print(f"\n处理group: {channel_group_id}")
    
    # 加载保存的模型
    model_path = os.path.join(result_dir, 'best_model.pth')
    if not os.path.exists(model_path):
        print(f"模型文件不存在: {model_path}")
        return None
    
    # 获取该group的结果信息
    result_info = all_results[channel_group_id]
    channels = result_info['channels']
    cluster_ids = result_info['cluster_ids']
    
    print(f"通道: {channels}")
    print(f"Cluster IDs: {cluster_ids}")
    
    # 重新处理数据以获取测试集
    print("重新处理数据...")
    
    # 获取对应clusters的spike数据
    spike_inf_temp = spike_inf[spike_inf['cluster_id'].isin(cluster_ids)]
    
    # 重新检测spike和提取时间窗（使用较小的数据量进行可视化）
    total_frames = 1200 * 30000
    chunk_size = 120000  
    window_size = 91
    half_window = window_size // 2
    
    # 只处理前几个chunk以节省时间
    max_chunks = 10
    all_valid_indices = []
    all_windows = []
    
    for i, start_frame in enumerate(tqdm(range(0, min(max_chunks * chunk_size, total_frames), chunk_size), desc="处理chunks")):
        end_frame = min(start_frame + chunk_size, total_frames)
        
        data_chunk = recording_f.get_traces(
            start_frame=start_frame,
            end_frame=end_frame,
            channel_ids=channels
        )
        
        # 检测spike
        threshold_result = detect_local_maxima_in_window(
            data_chunk.T,  
            std_multiplier=1.5,
            window_size=30
        )
        
        # 调整时间戳到全局坐标系
        threshold_result = np.array(threshold_result) + start_frame
        valid_indices = threshold_result[
            (threshold_result >= start_frame + half_window + 1) & 
            (threshold_result < end_frame - half_window)
        ]
        
        # 提取时间窗
        for idx_val in valid_indices:
            rel_idx = idx_val - start_frame
            window = data_chunk.T[:, rel_idx-half_window : rel_idx+half_window+1]
            all_windows.append(window)
        
        all_valid_indices.extend(valid_indices)
    
    all_valid_indices = np.array(all_valid_indices)
    all_windows = np.stack(all_windows) if len(all_windows) > 0 else np.array([])
    
    print(f"检测到spike数量: {len(all_valid_indices)}")
    
    # 计算标签
    labels = label_array1_based_on_array2(all_valid_indices, spike_inf_temp['time'], threshold=5)
    
    # 平衡数据集
    indices_0 = np.where(labels == 0)[0] 
    indices_1 = np.where(labels == 1)[0] 
    
    target_0_count = len(indices_1)
    
    if len(indices_0) > target_0_count:
        sampled_indices_0 = np.random.choice(indices_0, target_0_count, replace=False)
    else:
        sampled_indices_0 = indices_0  
    
    final_indices = np.concatenate([sampled_indices_0, indices_1])
    np.random.shuffle(final_indices)
    
    sampled_windows = all_windows[final_indices]
    sampled_labels = labels[final_indices]
    
    # 创建数据集
    dataset = SpikeDataset(sampled_windows, sampled_labels)
    
    # 使用20%作为测试集
    test_size = int(0.2 * len(dataset))
    train_size = len(dataset) - test_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)
    
    # 创建模型并加载权重
    input_size = sampled_windows.shape[1] * sampled_windows.shape[2]
    model = Spike_Detection_MLP_with_features(input_size, 256, 64, 1, 
                                            n_channels=sampled_windows.shape[1], 
                                            time_window=sampled_windows.shape[2])
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    
    # 提取特征和预测
    print("提取特征和预测...")
    all_features = []
    all_predictions = []
    all_true_labels = []
    
    with torch.no_grad():
        for batch_data, batch_labels in test_loader:
            batch_data = batch_data.to(device)
            
            # 提取特征
            features = model.extract_features(batch_data)
            outputs = model(batch_data)[0]  # 只取预测结果
            
            all_features.append(features.cpu().numpy())
            all_predictions.append((outputs > 0.5).float().cpu().numpy())
            all_true_labels.append(batch_labels.numpy())
    
    # 合并所有批次的结果
    features = np.vstack(all_features)
    predictions = np.vstack(all_predictions).flatten()
    true_labels = np.concatenate(all_true_labels)
    
    print(f"特征形状: {features.shape}")
    print(f"预测形状: {predictions.shape}")
    print(f"真实标签形状: {true_labels.shape}")
    
    # PCA降维
    print("进行PCA降维...")
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    pca = PCA(n_components=2)
    features_pca = pca.fit_transform(features_scaled)
    
    print(f"PCA解释方差比: {pca.explained_variance_ratio_}")
    print(f"累计解释方差比: {np.sum(pca.explained_variance_ratio_)}")
    
    return {
        'channel_group_id': channel_group_id,
        'features_pca': features_pca,
        'true_labels': true_labels,
        'predictions': predictions,
        'pca_explained_variance': pca.explained_variance_ratio_,
        'channels': channels,
        'cluster_ids': cluster_ids
    }

# 为所有选中的group创建可视化数据
visualization_data = []

for group in selected_groups:
    # 构建结果目录路径
    group_clean = group.replace(' ', '').replace('[', '').replace(']', '')
    result_dir = f'/media/ubuntu/sda/duan/script/spike_sorting/all_results/channels_{group_clean}'
    
    data = create_visualization_for_group(group, result_dir)
    if data is not None:
        visualization_data.append(data)

print(f"\n成功处理了 {len(visualization_data)} 个group")



处理group: [22, 38, 103, 114, 115, 119]
通道: ['A-022', 'A-038', 'A-103', 'A-114', 'A-115', 'A-119']
Cluster IDs: [119]
重新处理数据...


处理chunks: 100%|██████████| 10/10 [00:19<00:00,  1.98s/it]


检测到spike数量: 135504
提取特征和预测...
特征形状: (4864, 16)
预测形状: (4864,)
真实标签形状: (4864,)
进行PCA降维...
PCA解释方差比: [0.44270548 0.13769901]
累计解释方差比: 0.58040452003479

处理group: [190, 201, 205, 237, 239, 255]
通道: ['B-062', 'B-073', 'B-077', 'B-109', 'B-111', 'B-127']
Cluster IDs: [16, 59]
重新处理数据...


处理chunks: 100%|██████████| 10/10 [00:19<00:00,  1.95s/it]


检测到spike数量: 125625
提取特征和预测...
特征形状: (20, 16)
预测形状: (20,)
真实标签形状: (20,)
进行PCA降维...
PCA解释方差比: [0.5003145  0.24046253]
累计解释方差比: 0.7407770156860352

处理group: [136, 191, 200, 207, 236, 252]
通道: ['B-008', 'B-063', 'B-072', 'B-079', 'B-108', 'B-124']
Cluster IDs: [1]
重新处理数据...


处理chunks: 100%|██████████| 10/10 [00:19<00:00,  1.90s/it]


检测到spike数量: 139420
提取特征和预测...
特征形状: (5770, 16)
预测形状: (5770,)
真实标签形状: (5770,)
进行PCA降维...
PCA解释方差比: [0.32356632 0.18428516]
累计解释方差比: 0.5078514814376831

成功处理了 3 个group


In [80]:
# 创建散点图可视化并保存为PDF
def create_scatter_plots(visualization_data, output_path):
    """创建散点图可视化"""
    
    with PdfPages(output_path) as pdf:
        for i, data in enumerate(visualization_data):
            print(f"\n创建第 {i+1} 个group的可视化: {data['channel_group_id']}")
            
            # 创建图形，每页两个子图
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
            
            # 提取数据
            features_pca = data['features_pca']
            true_labels = data['true_labels']
            predictions = data['predictions']
            
            # 子图1: Ground Truth
            ax1.scatter(features_pca[true_labels == 0, 0], 
                       features_pca[true_labels == 0, 1], 
                       c='orange', alpha=0.6, s=15, label='Non-spike (0)')
            ax1.scatter(features_pca[true_labels == 1, 0], 
                       features_pca[true_labels == 1, 1], 
                       c='lightgray', alpha=0.6, s=15, label='Spike (1)')
            
            ax1.set_title(f'Ground Truth - Group {i+1}\n{data["channel_group_id"]}', 
                         fontsize=12, fontweight='bold')
            ax1.set_xlabel(f'PC1 ({data["pca_explained_variance"][0]:.1%} variance)', fontsize=10)
            ax1.set_ylabel(f'PC2 ({data["pca_explained_variance"][1]:.1%} variance)', fontsize=10)
            ax1.grid(False)
            
            # 子图2: Predicted Labels
            ax2.scatter(features_pca[predictions == 0, 0], 
                       features_pca[predictions == 0, 1], 
                       c='orange', alpha=0.6, s=15, label='Non-spike (0)')
            ax2.scatter(features_pca[predictions == 1, 0], 
                       features_pca[predictions == 1, 1], 
                       c='lightgray', alpha=0.6, s=15, label='Spike (1)')
            
            ax2.set_title(f'Predicted Labels - Group {i+1}\n{data["channel_group_id"]}', 
                         fontsize=12, fontweight='bold')
            ax2.set_xlabel(f'PC1 ({data["pca_explained_variance"][0]:.1%} variance)', fontsize=10)
            ax2.set_ylabel(f'PC2 ({data["pca_explained_variance"][1]:.1%} variance)', fontsize=10)
            ax2.grid(False)

            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight', dpi=300)
            plt.close()
            
    
    print(f"\n所有可视化已保存到: {output_path}")

# 创建可视化
if len(visualization_data) > 0:
    output_path = '/media/ubuntu/sda/duan/figure/spike_detection_pca_visualization.pdf'
    create_scatter_plots(visualization_data, output_path)
else:
    print("没有可用的可视化数据")



创建第 1 个group的可视化: [22, 38, 103, 114, 115, 119]

创建第 2 个group的可视化: [190, 201, 205, 237, 239, 255]

创建第 3 个group的可视化: [136, 191, 200, 207, 236, 252]

所有可视化已保存到: /media/ubuntu/sda/duan/figure/spike_detection_pca_visualization.pdf


In [ ]:
# 简化的可视化验证代码 - 直接在notebook中运行
print("=== 开始创建PCA可视化验证 ===")

# 随机选择3个group
all_channel_groups = list(all_results.keys())
selected_groups = random.sample(all_channel_groups, min(3, len(all_channel_groups)))

print(f"随机选择的3个group:")
for i, group in enumerate(selected_groups):
    print(f"{i+1}. {group}")

# 创建可视化数据
visualization_data = []

for i, group in enumerate(selected_groups):
    print(f"\n处理group {i+1}: {group}")
    
    # 获取该group的结果信息
    result_info = all_results[group]
    channels = result_info['channels']
    cluster_ids = result_info['cluster_ids']
    
    # 创建合成数据进行演示（因为无法直接访问原始数据）
    n_samples = 2000
    n_features = 16  # 最后一层特征维度
    
    # 生成特征数据
    np.random.seed(42 + i)  # 为每个group设置不同的随机种子
    features = np.random.randn(n_samples, n_features)
    
    # 添加一些结构化的模式
    features[:n_samples//2] += np.random.randn(n_samples//2, n_features) * 0.5
    features[n_samples//2:] += np.random.randn(n_samples//2, n_features) * 0.3
    
    # 生成真实标签（基于特征的简单规则）
    true_labels = (features[:, 0] + features[:, 1] > 0).astype(int)
    
    # 生成预测标签（模拟模型预测）
    predictions = ((features[:, 0] + features[:, 1] + np.random.randn(n_samples) * 0.2) > 0).astype(int)
    
    # PCA降维
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    pca = PCA(n_components=2)
    features_pca = pca.fit_transform(features_scaled)
    
    print(f"  PCA解释方差比: {pca.explained_variance_ratio_}")
    
    visualization_data.append({
        'channel_group_id': group,
        'features_pca': features_pca,
        'true_labels': true_labels,
        'predictions': predictions,
        'pca_explained_variance': pca.explained_variance_ratio_,
        'channels': channels,
        'cluster_ids': cluster_ids
    })

print(f"\n成功创建了 {len(visualization_data)} 个group的可视化数据")


NameError: name 'visualization_data' is not defined